# Four-model ink ensemble on the extra segments (atlas + quick_patches_from_D)

Same models and ensemble as `villa_ensemble_inference.ipynb`; outputs go to `output/test_visualizations/villa_ensemble/extras/`.

- extras are rendered in the background (uint8, `ves_zarrs2/<id>.zarr` + `masks/extras/<id>.png`; log: `_ves_tmp/assemble_extras.log`); cell 3 shows progress and resumes it if it stopped
- cell 4 waits until the main notebook has finished, so the GPU is never shared
- the per-scroll cells render every assembled extra, skip finished ones (rerun = resume) and log failures without stopping
- uint8 renders are symlinked into `ves_zarrs_villa/`, not copied

In [ ]:
import gc
import importlib
import os
import shutil
import subprocess
import sys
import time

if sys.executable.startswith("/opt/villa-venv"):
    raise RuntimeError("select the system Python 3.10 kernel (/usr/bin/python3); /opt/villa-venv is only used "
                       "internally for model inference")

REPO = "/vesuvius"
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
import assemble_test_segments as ats_mod
from utils import villa_ensemble as ve
ve = importlib.reload(ve)

ZARR_DIR = os.getenv("VESUVIUS_ZARR_PATH", os.path.join(REPO, "ves_zarrs2"))
MASK_DIR = ats_mod.EXTRAS_MASK_DIR
OUT_DIR = os.path.join(REPO, "output", "test_visualizations", "villa_ensemble", "extras")
MAIN_OUT_DIR = os.path.dirname(OUT_DIR)
ASSEMBLY_LOG = os.path.join(REPO, "_ves_tmp", "assemble_extras.log")
MIN_FREE_GB = 30
DEPTHS = (0,)
WRITE_DEPTH = 0
DIRECTION = "forward"
COMPILE = False
TAG = f"d{WRITE_DEPTH:+d}"
TITLES = {"dnative": "dense_native (16k)", "hecate": "Hecate 9.6µm",
          "v8in-1447": "v8-in PHerc1447 LOO-w062", "ink9-43": "ink_9um seed 43"}
# main-notebook fragments (name -> zarr id); their ensemble PNGs mark it as finished
MAIN = {"auto_grown_20260814140748": 20260814140748, "auto_grown_20260717193517": 20260717193517,
        "auto_grown_20260720090842": 20260720090842, "auto_grown_20260723112922652_merged": 20260723112922,
        "auto_grown_20260921094413486": 20260921094413, "auto_grown_20260918132724424": 20260918132724,
        "auto_grown_20260922073234974": 20260922073234, "auto_grown_20260922161631422_merged": 20260922161631,
        "PHerc0800_z9264_w040_abf": 20260928000001, "PHerc0125_z15344_w040_abf": 20260928000002,
        "PHerc0211_z7312_w080_abf": 20260928000003, "PHerc0211_z5520_w040_abf": 20260928000004,
        "seg46527_PHerc0814": 20260226000000}

# (zid, mesh_sub, vol_base, vol_shape, set); zid = mesh folder name
EXTRAS = ats_mod.extra_fragments()
SCROLL_OF = {zid: mesh_sub.split(os.sep)[1] if set_name == "quick_patches_from_D" else zid.split("_")[0]
             for zid, mesh_sub, _, _, set_name in EXTRAS}


def is_assembled(zid):
    """zarr + mask present; the assembler writes the mask after the last layer"""
    zarray = os.path.join(ZARR_DIR, f"{zid}.zarr", ".zarray")
    mask = os.path.join(MASK_DIR, f"{zid}.png")
    return os.path.isfile(zarray) and os.path.isfile(mask) and os.path.getmtime(mask) >= os.path.getmtime(zarray)


def outputs(zid):
    return (os.path.join(OUT_DIR, f"{zid}_4models_{TAG}.png"), os.path.join(OUT_DIR, f"{zid}_ensemble_{TAG}.png"))


def free_gb():
    return shutil.disk_usage(REPO).free / 1e9



print(f"{len(EXTRAS)} extras:", {s: list(SCROLL_OF.values()).count(s) for s in sorted(set(SCROLL_OF.values()))},      f"| free {free_gb():.0f} GB | out {OUT_DIR}")

In [ ]:
# --- assembly status; resumes the background render (uint8, raw-chunk cache dropped per segment) if it stopped ---
RESUME_ASSEMBLY = True

running = subprocess.run(["pgrep", "-f", "EXTRAS_MASK_DIR"], capture_output=True).returncode == 0
done = [zid for zid, *_ in EXTRAS if is_assembled(zid)]
print(f"assembled {len(done)}/{len(EXTRAS)} | assembly running: {running} | free {free_gb():.0f} GB")
if RESUME_ASSEMBLY and not running and len(done) < len(EXTRAS):
    script = f"""
import sys; sys.path.insert(0, {REPO!r})
import assemble_test_segments as a
a._MIN_FREE_BYTES = int({MIN_FREE_GB}e9)
res = []
for zid, sub, vb, vs, st in a.extra_fragments():
    if a._LOW_DISK.is_set():
        print('[extras] disk floor reached; stopping', flush=True); break
    print(f'=== extra {{st}}/{{zid}}', flush=True)
    res.append(a.render_fragment(zid, sub, vb, vs, 32, a.ZARR_DIR, a.SCRIPT_DIR, a.DEFAULT_CHUNK_DEPTH, a.DEFAULT_CHUNK_Y,
                                 a.DEFAULT_CHUNK_X, mask_dir=a.EXTRAS_MASK_DIR, output_dtype=a.EXTRAS_DTYPE, drop_cache=True))
print('SUMMARY', sum(s.startswith('OK') for _, s in res), 'ok of', len(res), [r for r in res if not r[1].startswith('OK')], flush=True)
"""
    with open(ASSEMBLY_LOG, "a") as log:
        subprocess.Popen([sys.executable, "-u", "-c", script], stdout=log, stderr=subprocess.STDOUT, cwd=REPO,
                         start_new_session=True)
    print("started background assembly ->", ASSEMBLY_LOG)

In [ ]:
# --- wait for villa_ensemble_inference.ipynb to finish: all its ensemble PNGs exist, or no model process for 20 min ---
IDLE_MINUTES = 20


def driver_running():
    return subprocess.run(["pgrep", "-f", "villa_driver.py"], capture_output=True).returncode == 0


idle_since, last_log = None, 0.0
while True:
    missing = [n for n in MAIN if not os.path.isfile(os.path.join(MAIN_OUT_DIR, f"{n}_ensemble_{TAG}.png"))]
    busy = driver_running()
    if not missing and not busy:
        print("main notebook finished")
        break
    idle_since = None if busy else (idle_since or time.time())
    if idle_since and time.time() - idle_since > IDLE_MINUTES * 60:
        print(f"no model process for {IDLE_MINUTES} min; proceeding ({len(missing)} main figures missing: {missing})")
        break
    if time.time() - last_log > 600:
        print(time.strftime("%H:%M"), f"waiting: {len(missing)} main figures left, driver running: {busy}", flush=True)
        last_log = time.time()
    time.sleep(60)

In [ ]:
# --- redraw the main notebook's ensemble figures with the composite below (from cached maps, no inference) ---
for name, scroll_id in MAIN.items():
    try:
        _, ensemble, _ = ve.ensemble(str(scroll_id), DEPTHS, DIRECTION, WRITE_DEPTH)
        ve.save_ensemble_figure(str(scroll_id), ensemble, f"4-model ensemble (mean z-score)  {TAG} {DIRECTION}",
                                os.path.join(MAIN_OUT_DIR, f"{name}_ensemble_{TAG}.png"))
        del ensemble
        gc.collect()
        print("[redrawn]", name, flush=True)
    except Exception as exc:
        print(f"[redraw skipped] {name}: {exc}", flush=True)


In [ ]:
# --- rendering helpers ---
RESULTS = {}


def render_extra(zid):
    t0 = time.time()
    ve.ensure_villa_input(zid, ZARR_DIR)
    ve.run_models(zid, DEPTHS, [DIRECTION], compile_model=COMPILE)
    stretched, ensemble, stats = ve.ensemble(zid, DEPTHS, DIRECTION, WRITE_DEPTH)
    grid_path, ens_path = outputs(zid)
    ve.save_grid([(f"{TITLES[k]}  {TAG} {DIRECTION}", stretched[k]) for k, _ in ve.MODELS], 2, grid_path)
    ve.save_ensemble_figure(zid, ensemble, f"{zid}  4-model ensemble (mean z-score)  {TAG} {DIRECTION}", ens_path)
    del stretched, ensemble, stats
    gc.collect()
    return time.time() - t0


def render_scroll(scroll):
    """every assembled extra of one scroll; finished ones are skipped, failures logged without stopping"""
    zids = [zid for zid, *_ in EXTRAS if SCROLL_OF[zid] == scroll]
    for i, zid in enumerate(zids, 1):
        if all(os.path.isfile(p) for p in outputs(zid)):
            RESULTS[zid] = "done (cached)"
            continue
        if not is_assembled(zid):
            RESULTS[zid] = "not assembled"
            print(f"[{scroll} {i}/{len(zids)}] {zid}: not assembled, skipped", flush=True)
            continue
        print(f"[{scroll} {i}/{len(zids)}] {zid}", flush=True)
        try:
            RESULTS[zid] = f"OK {render_extra(zid):.0f}s"
        except Exception as exc:
            RESULTS[zid] = f"FAILED: {exc}"
            print(f"[FAILED] {zid}: {exc}", flush=True)
    counts = {k: sum(RESULTS.get(z, "").startswith(k) for z in zids) for k in ("OK", "done", "not", "FAILED")}
    print(f"[{scroll}] {counts}")

In [ ]:
# --- extras: PHerc0846A quick_patches_from_D (22 segments) ---
render_scroll("PHerc0846A")

In [ ]:
# --- extras: PHerc0211 (51 segments) ---
render_scroll("PHerc0211")

In [ ]:
# --- extras: PHerc0813 (46 segments) ---
render_scroll("PHerc0813")

In [ ]:
# --- extras: PHerc0800 (44 segments, 8.64µm) ---
render_scroll("PHerc0800")

In [ ]:
# --- extras: PHerc0125 (62 segments) ---
render_scroll("PHerc0125")

In [ ]:
# --- extras: PHerc0358 (2), PHerc0826 (2), PHerc0257 (1) ---
for scroll in ("PHerc0358", "PHerc0826", "PHerc0257"):
    render_scroll(scroll)

In [ ]:
# --- summary ---
from collections import Counter

status = {zid: RESULTS.get(zid, "done (cached)" if all(os.path.isfile(p) for p in outputs(zid))
                           else "assembled, not run" if is_assembled(zid) else "not assembled")
          for zid, *_ in EXTRAS}
print(Counter(s.split(" ")[0].rstrip(":") for s in status.values()))
for zid, s in status.items():
    if s.startswith(("FAILED", "not")):
        print(f"  {zid}: {s}")
tmp_gb = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(os.path.join(REPO, "_ves_tmp")) for f in fs) / 1e9
print(f"free {free_gb():.0f} GB | _ves_tmp {tmp_gb:.1f} GB | figures in {OUT_DIR}")